# LA A03c — Reverse 1:1 versus third-touch maker-entry proxies

**Exploratory comparison on the already-inspected July–September 2026 sample.**

We run three separate policies on the **same 452 A02 candidate IDs**, using A03b's selected structural target:

1. **REVERSE_TAKER_1TO1** — invert original A03b direction and swap symmetric stop/target barriers; same first recorded entry print.
2. **MAKER_TOUCH_3RD** — keep rejection direction, place a post-only *hypothetical* limit at the original level after the second completed 1m rejection; assume eligible fill on opposite-aggressor print touching limit.
3. **MAKER_THROUGH_3RD** — same, except require opposing-aggressor print at least one assumed tick *through* the limit.

**Do not mistake an aggTrade print for guaranteed queue fill.** The maker results are plausible eligibility simulations, not historical execution records. No BBO, queue depth, funding, portfolio capital or independent holdout. A03b result files are not overwritten.

**Order invalidation:** If the historical price reaches the frozen target or the mirrored stop before an eligible passive fill, the resting order is cancelled as a stale setup. This is a causal exploratory assumption and is configurable.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

REPO = Path.cwd().resolve()
while not (REPO / "src" / "data_engineering").is_dir():
    if REPO == REPO.parent:
        raise RuntimeError("Launch Jupyter from the repo or its subdirectory")
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a03c import VERSION, Config, run
assert VERSION == "LA-A03c-0.1.1", "Pull updates and restart kernel"
DATA_ROOT = binance_root(REPO)
OUT = REPO / "notebooks" / "Liquidity_Absorption" / "03c"
CFG = Config()
print("Version:", VERSION, "| Binance files:", DATA_ROOT)
print("A03c has two maker ELIGIBILITY proxies, NOT recorded passive fills")


Version: LA-A03c-0.1.1 | Binance files: D:\MarketData\Binance
A03c has two maker ELIGIBILITY proxies, NOT recorded passive fills


In [2]:
decision = run(REPO, DATA_ROOT, OUT, CFG)
print("Version:", decision["version"], "| candidates:", decision["candidates"], "| daily trade files read:", decision["tape"]["trade_daily_files_read"])
print("Missing tape days:", decision["tape"]["missing_daily_files"])


KeyError: 'bar_close'

## Counts, fees and EV **per closed trade AND per 452 signals**

`mean_net_bp_per_signal_zero_if_no_trade_PARTIAL` includes zero realized P&L for genuinely unfilled/cancelled signals and divides by 452 **per market combined** only if recomputed across both assets. The per-symbol coverage files use 212 BTC / 240 ETH denominators. It is *partial* if any trades are censored; do not treat it as full EV.

The fee sensitivity below reprices the exact same assumed crossings. It **does not** estimate maker fill probability or compensate for missing quotes, execution delays, funding or queuing.


In [ ]:
trades = pd.read_csv(OUT / "03c_events.csv")
coverage = pd.read_csv(OUT / "03c_coverage.csv")
fee = pd.read_csv(OUT / "03c_fee_sensitivity.csv")
assert len(trades) == 452 * 3 and trades.groupby("a02_candidate_row").size().eq(3).all()
display(coverage)
print("All signal outcomes:")
display(trades.groupby(["symbol", "mode", "reason"]).size().rename("n").reset_index())
print("Descriptive total net bp per ORIGINAL signal — NOT full EV if censored:")
report = trades.groupby("mode").apply(lambda x: pd.Series({
    "signals":len(x), "entered":x.entry_time_ms.notna().sum(), "closed":x.reason.isin(["STOP", "TARGET", "TIMEOUT"]).sum(),
    "censored":x.reason.str.startswith("CENSORED").sum(),
    "mean_closed_net_bp":x.net_return_bp_assumed_costs.mean(),
    "partial_mean_net_bp_per_signal":x.net_return_bp_assumed_costs.sum() / len(x)}), include_groups=False)
display(report)
print("Fees only (bp), same candidate/fill scenarios; gross/funding/queue unchanged")
display(fee)


## Gross and net historical *virtual P&L* curves

These paths cumulate closed trade outcomes in exit-time order at **USDT $10,000 hypothetical notional per trade**. They are not invested account equity and allow overlapping hypothetical orders. A maker 'fill' is not guaranteed, even when historical price crosses through the resting level. **Zero-filled cases show no P&L**, but must be counted in the full opportunity denominator. Evaluate censor categories separately.


In [ ]:
for filename in ("03c_net_equity.png", "03c_gross_equity.png"):
    file = OUT / filename
    if file.exists():
        print(filename)
        display(Image(filename=str(file)))


## Research interpretation gate

- Do **not** conclude that reverse direction is profitable simply because original A03b lost money after costs.
- Do **not** claim fill probabilities for the optimistic/strict maker proxies without historical queue state, best bid/ask and own placement acknowledgement.
- A maker touch is the third market **revisit** after the second closed rejection; it is not a retrospectively confirmed third rejection candle. The passive order is placed earlier and waits.
- `2bp maker / 6bp taker` is an assumed fee mix for direct comparability with the older A03. Binance's advertised regular-user USDT-M rate is commonly 2bp maker / 5bp taker, with account-specific differences. Check actual historical account-tier, BNB discounts and promotions.
- A03b target levels remain frozen; passive entry at the level changes both timing and risk geometry. No LVN target, funding or capital/margin constraints.
- Keep all CSV/JSON/PNG outputs and the executed notebook in `03c/`; use them to audit gross/net/conditional and unconditional opportunity EV **before** modifying any thresholds.
